# Lab Assignment 3 - Task 1

**Name:** Akshat  
**Roll number:** 12340160

## Part 1: Parsing a Sentence with CKY

Given the following grammar in CNF.
1. Write the code for CKY algorithm and verify if the **sentence** follows or can be parsed by this grammar or not.
2. Also print the final CKY chart/table.

In [9]:
import nltk
from nltk.parse import ChartParser
from nltk.grammar import CFG

In [10]:
cky_grammar = CFG.fromstring("""
    S -> NP VP
    NP -> Det N
    VP -> V NP
    Det -> 'the'
    N -> 'man' | 'dog'
    V -> 'saw'
""")

sentence = "the man saw the dog"

### Approach

CKY fills a table where `chart[i][j]` holds every non terminal that can generate the words from
position `i` up to position `j`. The grammar is already in CNF, so every rule is either
`A -> B C` or `A -> 'word'`.

1. Read the rules out of the NLTK grammar object and split them into binary rules and lexical rules.
2. Fill the diagonal cells `chart[i][i+1]` using the lexical rules, one word at a time.
3. For spans of length 2 and above, try every split point `k`. If `B` is in `chart[i][k]` and `C` is
   in `chart[k][j]`, then add `A` to `chart[i][j]` for every rule `A -> B C`.
4. The sentence is in the language if the start symbol `S` ends up in `chart[0][n]`.

In [11]:
def build_rules(grammar):
    """Split the grammar productions into binary rules and lexical rules."""
    binary = []
    lexical = []
    for production in grammar.productions():
        rhs = production.rhs()
        if len(rhs) == 2:
            binary.append((production.lhs().symbol(), rhs[0].symbol(), rhs[1].symbol()))
        elif len(rhs) == 1 and isinstance(rhs[0], str):
            lexical.append((production.lhs().symbol(), rhs[0]))
    return binary, lexical


binary_rules, lexical_rules = build_rules(cky_grammar)
print("binary rules:", binary_rules)
print("lexical rules:", lexical_rules)

binary rules: [('S', 'NP', 'VP'), ('NP', 'Det', 'N'), ('VP', 'V', 'NP')]
lexical rules: [('Det', 'the'), ('N', 'man'), ('N', 'dog'), ('V', 'saw')]


In [12]:
def cky_parse(words, grammar, start="S"):
    """Run CKY on a list of words. Returns the chart and whether the sentence parses."""
    binary, lexical = build_rules(grammar)
    n = len(words)

    # chart[i][j] holds the categories covering words i .. j-1
    chart = [[set() for _ in range(n + 1)] for _ in range(n)]

    # step 1: the words themselves
    for i, word in enumerate(words):
        for lhs, terminal in lexical:
            if terminal == word:
                chart[i][i + 1].add(lhs)

    # step 2: longer and longer spans
    for length in range(2, n + 1):
        for i in range(n - length + 1):
            j = i + length
            for k in range(i + 1, j):            # every split point
                for lhs, b, cc in binary:
                    if b in chart[i][k] and cc in chart[k][j]:
                        chart[i][j].add(lhs)

    return chart, start in chart[0][n]


words = sentence.split()
chart, accepted = cky_parse(words, cky_grammar)

print("Sentence:", sentence)
print("Can be parsed?", accepted)

Sentence: the man saw the dog
Can be parsed? True


In [13]:
def print_chart(chart, words):
    """Print the CKY table. Row i is the start position, column j is the end position."""
    n = len(words)
    width = 12

    print("words:", "  ".join(f"{i}:{w}" for i, w in enumerate(words)))
    print()
    print(" " * 6 + "".join(f"{'j=' + str(j):^{width}}" for j in range(1, n + 1)))
    for i in range(n):
        row = f"i={i}  "
        for j in range(1, n + 1):
            if j <= i:
                row += " " * width
            else:
                cell = ",".join(sorted(chart[i][j])) or "."
                row += f"{cell:^{width}}"
        print(row)


print_chart(chart, words)

words: 0:the  1:man  2:saw  3:the  4:dog

          j=1         j=2         j=3         j=4         j=5     
i=0      Det          NP          .           .           S      
i=1                   N           .           .           .      
i=2                               V           .           VP     
i=3                                          Det          NP     
i=4                                                       N      


In [14]:
# the same chart written out span by span, which is easier to read
for i in range(len(words)):
    for j in range(i + 1, len(words) + 1):
        if chart[i][j]:
            span = " ".join(words[i:j])
            cell = ", ".join(sorted(chart[i][j]))
            print(f'chart[{i}][{j}] = {{{cell}}}  covers "{span}"')

chart[0][1] = {Det}  covers "the"
chart[0][2] = {NP}  covers "the man"
chart[0][5] = {S}  covers "the man saw the dog"
chart[1][2] = {N}  covers "man"
chart[2][3] = {V}  covers "saw"
chart[2][5] = {VP}  covers "saw the dog"
chart[3][4] = {Det}  covers "the"
chart[3][5] = {NP}  covers "the dog"
chart[4][5] = {N}  covers "dog"


The final cell `chart[0][5]` contains `S`, so the sentence **the man saw the dog** can be parsed
by this grammar.

## Part 2: Comparision with NLTK's chart parser
After implementing your CKY algorithm, compare its output for the given sentence with the output of NLTK's built-in chart parser using the `cky_grammar`. This comparison will help you verify if your implementation is correct.

In [15]:
parser = ChartParser(cky_grammar)
trees = list(parser.parse(words))

print("Number of parse trees found by NLTK:", len(trees))
for tree in trees:
    print(tree)
    tree.pretty_print()

Number of parse trees found by NLTK: 1
(S (NP (Det the) (N man)) (VP (V saw) (NP (Det the) (N dog))))
             S             
      _______|___           
     |           VP        
     |        ___|___       
     NP      |       NP    
  ___|___    |    ___|___   
Det      N   V  Det      N 
 |       |   |   |       |  
the     man saw the     dog



In [16]:
# compare the two results
my_answer = accepted
nltk_answer = len(trees) > 0

print(f"{'parser':<22}{'sentence accepted':>20}")
print("-" * 42)
print(f"{'my CKY':<22}{str(my_answer):>20}")
print(f"{'NLTK ChartParser':<22}{str(nltk_answer):>20}")
print("-" * 42)
print("\nDo the two agree?", my_answer == nltk_answer)

parser                   sentence accepted
------------------------------------------
my CKY                                True
NLTK ChartParser                      True
------------------------------------------

Do the two agree? True


### Comparison

Both parsers accept the sentence, so the implementation is correct on this input.

The difference is in what each one returns. My CKY only records *which* categories cover each span,
so it answers the yes or no question and shows the table. NLTK's `ChartParser` keeps the back
pointers as well, so it can print the actual tree:

```text
(S (NP (Det the) (N man)) (VP (V saw) (NP (Det the) (N dog))))
```

That tree matches the chart exactly. `chart[0][2]` holds `NP` for *the man*, `chart[2][5]` holds
`VP` for *saw the dog*, and the rule `S -> NP VP` joins them into the `S` sitting in `chart[0][5]`.
NLTK also finds only one tree, which agrees with the chart, since no cell ever received the same
category from two different split points.